# Session 5: Build an Internship Tracker feature

**BGS College workshop / V Naga Sai Rahul / Student notebook**

Start with a working career-practice app. Inspect its behavior, plan one feature, review the proposed diff, apply it, run checks, preview and save evidence. Your feature is **company and role search combined with status filtering**.

Work in pairs for 75 minutes. Save your own notebook copy, choose a CPU runtime and run sections in order. All companies and applications are fictional. There are no real applications, accounts, messages or backend services. Core work needs no API key or model. Use a prepared assistant only if available; manual editing is a complete route.

A fresh **Run all** finishes cleanly: 16 baseline tests pass, the search feature remains **PENDING**, and UI claims stay unchecked until you actually perform them.


## 1. Prepare the working starter

**What:** Check Node.js and create the embedded project in a dedicated `bgs-internship-lab` runtime folder.

**Why:** Colab uses a temporary cloud machine. No clone, provider setup, package download or public tunnel is needed. Node.js 18.20+ is supported here; use Node.js 20+ for the laptop route in README.md.

**Expected:** The runtime version and created-file count. Rerunning creates missing files and preserves existing edits. To begin again, first save/download your work and set `PROJECT_FOLDER` to a new unused simple folder name. There is no force-wipe switch.

**Read the code:** The supplied assets are embedded below. `run_tests` uses a 30-second Node process and fingerprints the original checks. All generated evidence and ZIP files stay inside the dedicated project folder.


In [ ]:
# @title Prepare the project (preserve existing edits)
# Standard-library helpers keep this lab independent of package installations.
import difflib
import hashlib
import html
import json
import re
import shutil
import subprocess
import zipfile
from pathlib import Path
from IPython.display import HTML, FileLink, display

# Check the prepared runtime before creating any project files.
NODE = shutil.which('node')
if NODE is None:
    raise RuntimeError('Node.js 18.20+ is required. Ask for the prepared runtime or use the laptop starter; this notebook installs nothing.')
node_version = subprocess.run([NODE, '--version'], capture_output=True, text=True, check=True, timeout=10).stdout.strip()
if tuple(int(part) for part in node_version.lstrip('v').split('.')[:3]) < (18, 20, 0):
    raise RuntimeError(f'Found {node_version}; use the prepared Node.js 18.20+ runtime or a Node.js 20+ laptop.')

# Every starter file is included, so this notebook can run without a repository clone.
ASSETS = {'.gitignore': '.env\n.env.*\nnode_modules/\n*.log\nBGS_Session_05_Your_Project.zip\n',
 'AGENTS.md': '# Session 5 / Internship Tracker\n'
              '\n'
              'This is a fictional career-practice app. It records invented applications on the current '
              'page. There are no real applications, messages, accounts or backend services.\n'
              '\n'
              '- Work only in this workshop project folder. Use plain HTML, CSS, JavaScript and Node '
              'built-ins; add no package dependencies.\n'
              '- The supplied add, validation, known-status update, status filter and summary features work. '
              'Preserve them and keep tracker.test.js unchanged.\n'
              '- Build one new feature: search company OR role text, trimmed and case-insensitive, combined '
              'with the existing status filter. Empty search keeps all status-matching records. A missing '
              'match shows the existing empty state.\n'
              "- Add/export filterApplications(applications, {status = 'All', search = ''} = {}) in "
              'tracker.js. Keep filterByStatus available. Reject non-text search with "Search must be text" '
              'in the message.\n'
              '- Add a labeled search control in index.html. In app.js, connect its input event to rendering '
              'and use both filter values. Render user text with textContent or createTextNode.\n'
              '- Workflow: inspect files and baseline -> brief plan -> propose a feature diff -> human '
              'reviews the exact diff -> apply -> run baseline plus new checks -> preview -> save evidence.\n'
              '- Baseline: node --test tracker.test.js (or npm test), expected 16 passing tests. New search '
              'checks remain PENDING until the feature is explicitly applied; then run node --test '
              'tracker.test.js search.feature-check.mjs (or npm run test:feature), expected 22 passes.\n'
              '- Laptop app: node server.mjs (or npm start), then http://127.0.0.1:8085. No npm install is '
              'needed.\n'
              '- Report only observed tests and browser interactions. Record the actual assistant route, or '
              'say manual editing/no model used. No model/API key is needed for core work.\n'
              '- Keep personal applications and credentials out of files, prompts, screenshots and evidence. '
              'Ask before changes outside this folder.\n',
 'CLAUDE.md': '# Session 5 / Internship Tracker\n'
              '\n'
              'This is a fictional career-practice app. It records invented applications on the current '
              'page. There are no real applications, messages, accounts or backend services.\n'
              '\n'
              '- Work only in this workshop project folder. Use plain HTML, CSS, JavaScript and Node '
              'built-ins; add no package dependencies.\n'
              '- The supplied add, validation, known-status update, status filter and summary features work. '
              'Preserve them and keep tracker.test.js unchanged.\n'
              '- Build one new feature: search company OR role text, trimmed and case-insensitive, combined '
              'with the existing status filter. Empty search keeps all status-matching records. A missing '
              'match shows the existing empty state.\n'
              "- Add/export filterApplications(applications, {status = 'All', search = ''} = {}) in "
              'tracker.js. Keep filterByStatus available. Reject non-text search with "Search must be text" '
              'in the message.\n'
              '- Add a labeled search control in index.html. In app.js, connect its input event to rendering '
              'and use both filter values. Render user text with textContent or createTextNode.\n'
              '- Workflow: inspect files and baseline -> brief plan -> propose a feature diff -> human '
              'reviews the exact diff -> apply -> run baseline plus new checks -> preview -> save evidence.\n'
              '- Baseline: node --test tracker.test.js (or npm test), expected 16 passing tests. New search '
              'checks remain PENDING until the feature is explicitly applied; then run node --test '
              'tracker.test.js search.feature-check.mjs (or npm run test:feature), expected 22 passes.\n'
              '- Laptop app: node server.mjs (or npm start), then http://127.0.0.1:8085. No npm install is '
              'needed.\n'
              '- Report only observed tests and browser interactions. Record the actual assistant route, or '
              'say manual editing/no model used. No model/API key is needed for core work.\n'
              '- Keep personal applications and credentials out of files, prompts, screenshots and evidence. '
              'Ask before changes outside this folder.\n',
 'GEMINI.md': '# Session 5 / Internship Tracker\n'
              '\n'
              'This is a fictional career-practice app. It records invented applications on the current '
              'page. There are no real applications, messages, accounts or backend services.\n'
              '\n'
              '- Work only in this workshop project folder. Use plain HTML, CSS, JavaScript and Node '
              'built-ins; add no package dependencies.\n'
              '- The supplied add, validation, known-status update, status filter and summary features work. '
              'Preserve them and keep tracker.test.js unchanged.\n'
              '- Build one new feature: search company OR role text, trimmed and case-insensitive, combined '
              'with the existing status filter. Empty search keeps all status-matching records. A missing '
              'match shows the existing empty state.\n'
              "- Add/export filterApplications(applications, {status = 'All', search = ''} = {}) in "
              'tracker.js. Keep filterByStatus available. Reject non-text search with "Search must be text" '
              'in the message.\n'
              '- Add a labeled search control in index.html. In app.js, connect its input event to rendering '
              'and use both filter values. Render user text with textContent or createTextNode.\n'
              '- Workflow: inspect files and baseline -> brief plan -> propose a feature diff -> human '
              'reviews the exact diff -> apply -> run baseline plus new checks -> preview -> save evidence.\n'
              '- Baseline: node --test tracker.test.js (or npm test), expected 16 passing tests. New search '
              'checks remain PENDING until the feature is explicitly applied; then run node --test '
              'tracker.test.js search.feature-check.mjs (or npm run test:feature), expected 22 passes.\n'
              '- Laptop app: node server.mjs (or npm start), then http://127.0.0.1:8085. No npm install is '
              'needed.\n'
              '- Report only observed tests and browser interactions. Record the actual assistant route, or '
              'say manual editing/no model used. No model/API key is needed for core work.\n'
              '- Keep personal applications and credentials out of files, prompts, screenshots and evidence. '
              'Ask before changes outside this folder.\n',
 'README.md': '# Internship Tracker / Session 5\n'
              '\n'
              'Build a useful feature on top of a working application. The three fictional companies are '
              'Aster Labs, Cedar Systems and Orbit Works. Each record has an application ID, company, role, '
              'ISO deadline and one of Saved, Applied, Interview, Offer or Closed.\n'
              '\n'
              '## Local setup\n'
              '\n'
              'Use Node.js 20 or later on a laptop and a modern browser. The Colab companion supports its '
              'preflighted Node.js 18.20+ runtime. Run `node --version` first. No packages, API keys, '
              'accounts or installation commands are required for this project.\n'
              '\n'
              'Open a terminal in this folder:\n'
              '\n'
              '```text\n'
              'node --test tracker.test.js\n'
              'node server.mjs\n'
              '```\n'
              '\n'
              'The fresh starter passes all **16 baseline tests**. Open http://127.0.0.1:8085 and keep that '
              'server terminal open. Use another terminal for checks. `npm test` and `npm start` are '
              'optional aliases; `npm install` is unnecessary. The server binds only to 127.0.0.1.\n'
              '\n'
              '## Working baseline\n'
              '\n'
              'Add a fictional application, reject missing fields/duplicate IDs/invalid dates, update an '
              'existing application to a known status, filter by status, and inspect total/per-status '
              'counts. Counts summarize the full list, including applications hidden by the current filter. '
              'Data lives in page memory: reloading restores the three examples. No real application is '
              'submitted; there is no login, email or backend.\n'
              '\n'
              '## Student feature: company and role search\n'
              '\n'
              'Add a labeled search input. Match a substring in company **or** role, ignore letter case, and '
              'trim surrounding whitespace. Combine search with the selected status. Blank search preserves '
              'the status filter. A missing match uses the empty state. Preserve add/update/validation/count '
              'behavior and literal text rendering.\n'
              '\n'
              "Export `filterApplications(applications, {status = 'All', search = ''} = {})` from tracker.js "
              'and use it in app.js. Keep `filterByStatus` available. Reject a non-text search with a '
              '`Search must be text` message. The feature requirements and explicit checks are in '
              'search.feature-check.mjs; that file is deliberately outside the default baseline test command '
              'because search is new work.\n'
              '\n'
              'Follow **inspect -> plan -> review the proposed feature diff -> apply -> tests -> preview -> '
              'save**. TASK_PROMPTS.md contains copyable prompts for an optional prepared coding assistant '
              'or browser assistant. Manual editing is equally valid. Record which route actually ran; this '
              'project calls no model.\n'
              '\n'
              'After applying your reviewed change:\n'
              '\n'
              '```text\n'
              'node --test tracker.test.js search.feature-check.mjs\n'
              '```\n'
              '\n'
              'Expect **22 passes**: 16 unchanged baseline checks plus 6 search checks. Test the UI '
              'separately: search `DATA`, combine `intern` with Interview, try an empty result, clear the '
              'search, then add and update a fictional record. Test keyboard access. Node logic checks do '
              'not establish browser behavior.\n'
              '\n'
              '## Evidence and timing\n'
              '\n'
              'Save the proposed/reviewed diff, actual baseline and final output, observed UI checks, actual '
              'assistant route and one limitation. In Colab, the default Run All leaves the new feature '
              '**PENDING** and UI checkboxes unclaimed; edit the source proposal and explicitly apply it '
              'after review. Download the project ZIP and save a copy of the notebook before the temporary '
              'runtime ends. Rerunning setup preserves existing files. To start fresh, use an unused '
              'project-folder name in setup; never overwrite your only edited copy.\n'
              '\n'
              'Suggested 75 minutes: 10 inspect/baseline/preview, 10 plan, 20 feature proposal and review, '
              '15 apply/tests, 15 browser checks, 5 save/explain.\n',
 'TASK_PROMPTS.md': '# Session 5 / Feature-building prompts\n'
                    '\n'
                    'Use only this fictional workshop project. An assistant is optional; manual editing '
                    'follows the same workflow. Keep normal tool approvals enabled. Do not paste personal '
                    'applications or credentials into a prompt.\n'
                    '\n'
                    '## Inspect and plan\n'
                    '\n'
                    'Read AGENTS.md, tracker.js, app.js, index.html, tracker.test.js and '
                    'search.feature-check.mjs. Run the 16 baseline checks and describe the working '
                    'add/status/filter/summary behavior. Do not edit yet. Plan one feature: '
                    'case-insensitive, trimmed substring search across company OR role, combined with the '
                    'status filter. Explain the files and checks needed. Add no dependencies.\n'
                    '\n'
                    '## Propose the feature diff for review\n'
                    '\n'
                    'Return a focused unified diff for tracker.js, app.js and index.html. Add/export '
                    "filterApplications(applications, {status = 'All', search = ''} = {}), preserve "
                    'filterByStatus, and reject non-text queries with a Search must be text message. Add a '
                    'labeled search input, render on its input event and combine status and search. Keep '
                    'user strings literal through textContent/createTextNode. Preserve all baseline features '
                    'and tests. Do not apply the diff until I have reviewed it.\n'
                    '\n'
                    '## Review independently\n'
                    '\n'
                    'Review the proposed diff against the requirements without editing. Check company OR '
                    'role matching, mixed case, whitespace, status AND search, no results, blank search, '
                    'invalid query type, unchanged validation and full-list counts. Check selectors/labels '
                    'and safe text rendering. Identify concrete issues with file references, or state the '
                    'scope of the review.\n'
                    '\n'
                    '## Apply and verify\n'
                    '\n'
                    'Apply only the reviewed feature change. Show the actual diff and run node --test '
                    'tracker.test.js search.feature-check.mjs. Report actual pass/fail counts and any '
                    'command limitations. Do not delete tests or weaken assertions. Describe the browser '
                    'interactions still to be performed, including keyboard operation.\n'
                    '\n'
                    '## Browser assistant or manual route\n'
                    '\n'
                    'If a coding client is unavailable, give a browser assistant only the fictional source '
                    'and requirements above. Request a proposed diff, read it, apply it yourself with the '
                    'notebook source editor or local editor, and run the checks yourself. No provider, '
                    'router, account or API key is needed for manual work. Record the tool/provider/model '
                    'only if actually used; say manual edit/no model otherwise.\n'
                    '\n'
                    '## Explain and document\n'
                    '\n'
                    'Update your README/evidence with the search feature, actual commands/results, checked '
                    'browser behavior and a limitation. In thirty seconds, show the company match, role '
                    'match, combination with status, unchanged baseline checks and your reviewed diff. Say '
                    'which work you did and which route actually ran.\n',
 'app.js': 'import {STATUSES, DEMO_APPLICATIONS, addApplication, updateApplicationStatus, filterByStatus, '
           "summarizeApplications} from './tracker.js';\n"
           '\n'
           'let applications = DEMO_APPLICATIONS.map(record => ({...record}));\n'
           "const form = document.querySelector('#application-form');\n"
           "const list = document.querySelector('#application-list');\n"
           "const filter = document.querySelector('#status-filter');\n"
           "const summary = document.querySelector('#summary-counts');\n"
           "const resultCount = document.querySelector('#result-count');\n"
           "const message = document.querySelector('#message');\n"
           '\n'
           'function option(value) {\n'
           "  const item = document.createElement('option');\n"
           '  item.value = value;\n'
           '  item.textContent = value;\n'
           '  return item;\n'
           '}\n'
           '\n'
           "filter.append(option('All'), ...STATUSES.map(option));\n"
           "form.elements.namedItem('status').append(...STATUSES.map(option));\n"
           '\n'
           'function render() {\n'
           '  const totals = summarizeApplications(applications);\n'
           '  summary.replaceChildren();\n'
           "  for (const [label, count] of [['Total', totals.total], ...Object.entries(totals.counts)]) {\n"
           "    const item = document.createElement('div');\n"
           "    const number = document.createElement('strong');\n"
           "    const text = document.createElement('span');\n"
           '    number.textContent = String(count);\n'
           '    text.textContent = label;\n'
           '    item.append(number, text);\n'
           '    summary.append(item);\n'
           '  }\n'
           '  const visible = filterByStatus(applications, filter.value);\n'
           '  resultCount.textContent = `${visible.length} of ${applications.length} applications`;\n'
           '  list.replaceChildren();\n'
           '  if (!visible.length) {\n'
           "    const empty = document.createElement('p');\n"
           "    empty.className = 'empty';\n"
           "    empty.textContent = 'No applications match this view. Choose another filter or add a "
           "fictional application.';\n"
           '    list.append(empty);\n'
           '  }\n'
           '  for (const record of visible) {\n'
           "    const card = document.createElement('article');\n"
           "    card.className = 'application';\n"
           '    card.dataset.applicationId = record.id;\n'
           "    const details = document.createElement('div');\n"
           "    const company = document.createElement('h3');\n"
           "    const role = document.createElement('p');\n"
           "    const meta = document.createElement('p');\n"
           "    const deadline = document.createElement('time');\n"
           '    company.textContent = record.company;\n'
           "    role.className = 'role';\n"
           '    role.textContent = record.role;\n'
           "    meta.className = 'meta';\n"
           "    meta.append(document.createTextNode(record.id + ' / Deadline '));\n"
           '    deadline.dateTime = record.deadline;\n'
           '    deadline.textContent = record.deadline;\n'
           '    meta.append(deadline);\n'
           '    details.append(company, role, meta);\n'
           "    const label = document.createElement('label');\n"
           "    label.className = 'status-control';\n"
           "    const caption = document.createElement('span');\n"
           "    caption.textContent = 'Status';\n"
           "    const select = document.createElement('select');\n"
           "    select.setAttribute('aria-label', `Status for ${record.id}`);\n"
           '    select.append(...STATUSES.map(option));\n'
           '    select.value = record.status;\n'
           "    select.addEventListener('change', () => {\n"
           '      applications = updateApplicationStatus(applications, record.id, select.value);\n'
           '      message.textContent = `${record.id} status updated.`;\n'
           "      message.className = 'success';\n"
           '      render();\n'
           '    });\n'
           '    label.append(caption, select);\n'
           '    card.append(details, label);\n'
           '    list.append(card);\n'
           '  }\n'
           '}\n'
           '\n'
           "form.addEventListener('submit', event => {\n"
           '  event.preventDefault();\n'
           '  try {\n'
           '    const input = Object.fromEntries(new FormData(form).entries());\n'
           '    applications = addApplication(applications, input);\n'
           '    form.reset();\n'
           "    message.className = 'success';\n"
           "    message.textContent = 'Fictional application added. Choose All if your filter hides it.';\n"
           '    render();\n'
           '  } catch (problem) {\n'
           "    message.className = 'error';\n"
           '    message.textContent = problem.message;\n'
           '  }\n'
           '});\n'
           "filter.addEventListener('change', render);\n"
           'render();\n',
 'index.html': '<!doctype html>\n'
               '<html lang="en">\n'
               '<head><meta charset="utf-8"><meta name="viewport" '
               'content="width=device-width,initial-scale=1"><title>Internship Tracker / BGS '
               'Lab</title><link rel="icon" href="data:,"><link rel="stylesheet" href="styles.css"></head>\n'
               '<body>\n'
               '  <header><a class="brand" href="#">Internship Tracker</a><span>BGS College workshop / '
               'Session 5</span></header>\n'
               '  <main>\n'
               '    <section class="intro"><div><p class="context">Your next chapter, in view.</p><h1>One '
               'place for<br>your next move.</h1></div><p>Keep a simple record of opportunities, deadlines '
               'and progress. Practice with fictional applications in this workshop.</p></section>\n'
               '    <section aria-label="Application summary" id="summary-counts" '
               'class="summary"></section>\n'
               '    <div class="workspace">\n'
               '      <section aria-labelledby="applications-title"><div class="list-heading"><div><h2 '
               'id="applications-title">Applications</h2><p id="result-count" '
               'aria-live="polite"></p></div><label class="filter-label">Show status<select '
               'id="status-filter"></select></label></div><div id="application-list"></div></section>\n'
               '      <aside aria-labelledby="add-title"><h2 id="add-title">Add an application</h2><p '
               'class="hint">Use invented details. Each ID must be unique.</p>\n'
               '        <form id="application-form" novalidate>\n'
               '          <label>Application ID<input name="id" placeholder="APP-004" maxlength="24" '
               'required autocomplete="off"></label>\n'
               '          <label>Company<input name="company" placeholder="Example Studio" maxlength="80" '
               'required autocomplete="off"></label>\n'
               '          <label>Role<input name="role" placeholder="Engineering Intern" maxlength="80" '
               'required autocomplete="off"></label>\n'
               '          <label>Deadline<input name="deadline" type="date" required></label>\n'
               '          <label>Initial status<select name="status"></select></label>\n'
               '          <button type="submit">Add application</button>\n'
               '        </form>\n'
               '        <p id="message" role="status" aria-live="polite"></p><p class="hint">Practice only. '
               'This page sends nothing. Reloading restores the three examples.</p>\n'
               '      </aside>\n'
               '    </div>\n'
               '  </main>\n'
               '  <footer>V Naga Sai Rahul / BGS College workshop / Fictional companies and '
               'applications</footer>\n'
               '  <script type="module" src="app.js"></script>\n'
               '</body></html>\n',
 'package.json': '{\n'
                 '  "name": "bgs-internship-tracker",\n'
                 '  "private": true,\n'
                 '  "type": "module",\n'
                 '  "scripts": {\n'
                 '    "start": "node server.mjs",\n'
                 '    "test": "node --test tracker.test.js",\n'
                 '    "test:feature": "node --test tracker.test.js search.feature-check.mjs"\n'
                 '  },\n'
                 '  "engines": {\n'
                 '    "node": ">=18.20"\n'
                 '  }\n'
                 '}\n',
 'search.feature-check.mjs': '// Run explicitly after applying the reviewed search feature; npm test runs '
                             'the working baseline.\n'
                             "import test from 'node:test';\n"
                             "import assert from 'node:assert/strict';\n"
                             "import * as tracker from './tracker.js';\n"
                             'const rows = tracker.DEMO_APPLICATIONS.map(record => ({...record}));\n'
                             'function search(options) {\n'
                             "  assert.equal(typeof tracker.filterApplications, 'function', 'Export "
                             "filterApplications(applications, {status, search}) for this new feature.');\n"
                             '  return tracker.filterApplications(rows, options);\n'
                             '}\n'
                             "test('search matches a company substring without case sensitivity', () => {\n"
                             "  assert.deepEqual(search({search: 'aStEr'}).map(record => record.id), "
                             "['APP-001']);\n"
                             '});\n'
                             "test('search also matches role text and trims outer whitespace', () => {\n"
                             "  assert.deepEqual(search({search: '  DATA  '}).map(record => record.id), "
                             "['APP-002']);\n"
                             "  assert.equal(search({search: 'intern'}).length, 3);\n"
                             '});\n'
                             "test('empty and whitespace search preserve the status filter', () => {\n"
                             "  assert.deepEqual(search({search: '', status: 'Applied'}).map(record => "
                             "record.id), ['APP-002']);\n"
                             "  assert.equal(search({search: '   '}).length, 3);\n"
                             '});\n'
                             "test('search combines with status using both conditions', () => {\n"
                             "  assert.deepEqual(search({search: 'intern', status: 'Interview'}).map(record "
                             "=> record.id), ['APP-003']);\n"
                             "  assert.deepEqual(search({search: 'Cedar', status: 'Saved'}), []);\n"
                             '});\n'
                             "test('no match yields an empty list and search leaves input unchanged', () => "
                             '{\n'
                             '  const before = JSON.stringify(rows);\n'
                             "  assert.deepEqual(search({search: 'no-such-company'}), []);\n"
                             '  assert.equal(JSON.stringify(rows), before);\n'
                             '});\n'
                             "test('search preserves invalid-status validation and rejects a non-text "
                             "query', () => {\n"
                             "  assert.throws(() => search({status: 'Unknown', search: ''}), /known status "
                             'filter/);\n'
                             '  assert.throws(() => search({search: 42}), /Search must be text/);\n'
                             '});\n',
 'server.mjs': "import http from 'node:http';\n"
               "import fs from 'node:fs/promises';\n"
               "import path from 'node:path';\n"
               "import {fileURLToPath} from 'node:url';\n"
               'const root = path.dirname(fileURLToPath(import.meta.url));\n'
               'const files = new Map([\n'
               "  ['/', ['index.html', 'text/html; charset=utf-8']],\n"
               "  ['/index.html', ['index.html', 'text/html; charset=utf-8']],\n"
               "  ['/app.js', ['app.js', 'text/javascript; charset=utf-8']],\n"
               "  ['/tracker.js', ['tracker.js', 'text/javascript; charset=utf-8']],\n"
               "  ['/styles.css', ['styles.css', 'text/css; charset=utf-8']],\n"
               ']);\n'
               'const port = Number(process.env.PORT || 8085);\n'
               "if (!Number.isInteger(port) || port < 1024 || port > 65535) throw new Error('Choose a port "
               "from 1024 to 65535.');\n"
               'http.createServer(async (request, response) => {\n'
               "  const entry = files.get(new URL(request.url, 'http://127.0.0.1').pathname);\n"
               "  if (request.method !== 'GET' || !entry) { response.writeHead(404); response.end('Not "
               "found'); return; }\n"
               '  try {\n'
               '    const contents = await fs.readFile(path.join(root, entry[0]));\n'
               "    response.writeHead(200, {'Content-Type': entry[1], 'Cache-Control': 'no-store'});\n"
               '    response.end(contents);\n'
               "  } catch { response.writeHead(500); response.end('Could not read the local app file.'); }\n"
               "}).listen(port, '127.0.0.1', () => console.log(`Internship Tracker: http://127.0.0.1:${port} "
               '(local practice only)`));\n',
 'styles.css': ':root{color-scheme:light;--paper:#fafbff;--white:#fff;--ink:#192245;--body:#3e4967;--muted:#64708b;--line:#dce1ef;--indigo:#3645b3;--pale:#eef0ff;--green:#16634b;--error:#a32336}\n'
               '*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--body);font:16px/1.5 '
               '"Segoe '
               'UI",Arial,sans-serif}header{display:flex;align-items:center;justify-content:space-between;gap:20px;max-width:1260px;margin:auto;padding:24px '
               '36px;border-bottom:1px solid var(--line)}header '
               'span{font-size:13px;color:var(--muted)}.brand{font-size:20px;font-weight:700;color:var(--indigo);text-decoration:none}main{max-width:1260px;margin:auto;padding:40px '
               '36px '
               '60px}.intro{display:flex;justify-content:space-between;align-items:end;gap:50px;margin-bottom:34px}.context{font-size:15px;color:var(--indigo);margin:0 '
               '0 12px}h1{font:600 clamp(36px,5vw,60px)/1.02 "Segoe '
               'UI",Arial,sans-serif;letter-spacing:-.055em;color:var(--ink);margin:0}h2{font-size:23px;letter-spacing:-.025em;color:var(--ink);margin:0}h3{font-size:20px;color:var(--ink);margin:0 '
               '0 3px}.intro>p{max-width:340px;font-size:17px;margin:0 0 '
               '4px}.summary{display:grid;grid-template-columns:repeat(6,1fr);background:var(--indigo);color:white;border-radius:12px;padding:18px '
               '8px;margin-bottom:36px}.summary>div{padding:0 18px;border-right:1px solid '
               '#ffffff40}.summary>div:last-child{border:0}.summary '
               'strong{display:block;font-weight:600;font-size:29px;line-height:1.2}.summary '
               'span{font-size:13px;color:#e6e8ff}.workspace{display:grid;grid-template-columns:minmax(0,1.65fr) '
               'minmax(260px,1fr);gap:40px}.list-heading{display:flex;justify-content:space-between;align-items:center;gap:16px;margin-bottom:18px}#result-count{margin:4px '
               '0 '
               '0;color:var(--muted);font-size:13px}.filter-label{font-size:12px;color:var(--muted);min-width:140px}.filter-label '
               'select{margin-top:4px}.application{display:flex;align-items:center;justify-content:space-between;gap:20px;border-top:1px '
               'solid var(--line);padding:22px 0}.application:first-child{border-top:2px solid '
               'var(--ink)}.role{margin:0 0 '
               '11px}.meta{margin:0;color:var(--muted);font-size:12px}.status-control{width:128px;flex-shrink:0;font-size:12px;color:var(--muted)}.status-control '
               'select{margin-top:4px}aside{background:var(--white);border:1px solid '
               'var(--line);border-radius:12px;padding:25px;align-self:start}.hint{font-size:13px;color:var(--muted);margin:9px '
               '0 18px}form{display:grid;gap:13px}form '
               'label{font-size:13px;display:grid;gap:5px}input,select,button{font:inherit}input,select{width:100%;min-width:0;padding:9px '
               '11px;border:1px solid '
               '#bdc6df;border-radius:5px;background:white;color:var(--ink)}button{border:0;border-radius:5px;background:var(--indigo);color:white;padding:11px '
               '15px;font-weight:600;cursor:pointer;margin-top:5px}button:hover{background:#263494}:focus-visible{outline:3px '
               'solid #dd8b21;outline-offset:3px}#message{font-size:13px;min-height:1.5em;margin:15px '
               '0}.success{color:var(--green)}.error{color:var(--error)}.empty{padding:30px 0;border-top:2px '
               'solid var(--ink);color:var(--muted)}footer{max-width:1260px;margin:auto;border-top:1px solid '
               'var(--line);padding:20px '
               '36px;font-size:12px;color:var(--muted)}.search-label{display:grid;font-size:13px;gap:5px;margin-bottom:18px}\n'
               '.application>div{min-width:0;overflow-wrap:anywhere}\n'
               '@media(max-width:760px){header{padding:20px;align-items:start;flex-direction:column;gap:4px}main{padding:28px '
               '20px '
               '40px}.intro{display:block}.intro>p{margin-top:22px}.workspace{grid-template-columns:1fr;gap:30px}.summary{grid-template-columns:repeat(3,1fr);row-gap:18px}.summary>div:nth-child(3){border:0}.application{gap:12px}.status-control{width:110px}.application '
               'h3{font-size:18px}.meta{font-size:11px}aside{padding:22px}footer{padding:20px}}\n',
 'tracker.js': "export const STATUSES = Object.freeze(['Saved', 'Applied', 'Interview', 'Offer', "
               "'Closed']);\n"
               'export const DEMO_APPLICATIONS = Object.freeze([\n'
               "  Object.freeze({id: 'APP-001', company: 'Aster Labs', role: 'Frontend Intern', deadline: "
               "'2027-01-15', status: 'Saved'}),\n"
               "  Object.freeze({id: 'APP-002', company: 'Cedar Systems', role: 'Data Intern', deadline: "
               "'2027-01-20', status: 'Applied'}),\n"
               "  Object.freeze({id: 'APP-003', company: 'Orbit Works', role: 'Software Intern', deadline: "
               "'2027-01-25', status: 'Interview'}),\n"
               ']);\n'
               '\n'
               'export function validateApplication(input) {\n'
               "  if (!input || Array.isArray(input) || typeof input !== 'object') {\n"
               "    throw new TypeError('Application must be an object.');\n"
               '  }\n'
               '  const record = {};\n'
               "  for (const field of ['id', 'company', 'role', 'deadline', 'status']) {\n"
               "    if (typeof input[field] !== 'string') throw new TypeError(`${field} must be text.`);\n"
               '    record[field] = input[field].trim();\n'
               '  }\n'
               '  if (!/^[A-Z0-9][A-Z0-9-]{0,23}$/.test(record.id)) {\n'
               "    throw new RangeError('ID must use 1 to 24 uppercase letters, digits or hyphens.');\n"
               '  }\n'
               "  for (const field of ['company', 'role']) {\n"
               '    if (!record[field] || record[field].length > 80) {\n'
               '      throw new RangeError(`${field} must contain 1 to 80 characters.`);\n'
               '    }\n'
               '  }\n'
               "  const timestamp = Date.parse(record.deadline + 'T00:00:00Z');\n"
               '  if (!/^\\d{4}-\\d{2}-\\d{2}$/.test(record.deadline) || !Number.isFinite(timestamp)\n'
               '      || new Date(timestamp).toISOString().slice(0, 10) !== record.deadline) {\n'
               "    throw new RangeError('Deadline must be a real date in YYYY-MM-DD format.');\n"
               '  }\n'
               "  if (!STATUSES.includes(record.status)) throw new RangeError('Choose a known status.');\n"
               '  return record;\n'
               '}\n'
               '\n'
               'function readApplications(applications) {\n'
               "  if (!Array.isArray(applications)) throw new TypeError('Applications must be an array.');\n"
               '  const records = applications.map(validateApplication);\n'
               '  if (new Set(records.map(record => record.id)).size !== records.length) {\n'
               "    throw new RangeError('Application IDs must be unique.');\n"
               '  }\n'
               '  return records;\n'
               '}\n'
               '\n'
               'export function addApplication(applications, input) {\n'
               '  const records = readApplications(applications);\n'
               '  const record = validateApplication(input);\n'
               "  if (records.some(existing => existing.id === record.id)) throw new RangeError('This "
               "application ID already exists.');\n"
               '  return [...records, record];\n'
               '}\n'
               '\n'
               'export function updateApplicationStatus(applications, id, status) {\n'
               '  const records = readApplications(applications);\n'
               "  if (!STATUSES.includes(status)) throw new RangeError('Choose a known status.');\n"
               "  if (!records.some(record => record.id === id)) throw new RangeError('Unknown application "
               "ID.');\n"
               '  return records.map(record => record.id === id ? {...record, status} : record);\n'
               '}\n'
               '\n'
               "export function filterByStatus(applications, status = 'All') {\n"
               '  const records = readApplications(applications);\n'
               "  if (status !== 'All' && !STATUSES.includes(status)) throw new RangeError('Choose a known "
               "status filter.');\n"
               "  return status === 'All' ? records : records.filter(record => record.status === status);\n"
               '}\n'
               '\n'
               'export function summarizeApplications(applications) {\n'
               '  const records = readApplications(applications);\n'
               '  const counts = Object.fromEntries(STATUSES.map(status => [status, 0]));\n'
               '  for (const record of records) counts[record.status] += 1;\n'
               '  return {total: records.length, counts};\n'
               '}\n',
 'tracker.test.js': "import test from 'node:test';\n"
                    "import assert from 'node:assert/strict';\n"
                    'import {STATUSES, DEMO_APPLICATIONS, validateApplication, addApplication, '
                    "updateApplicationStatus, filterByStatus, summarizeApplications} from './tracker.js';\n"
                    'const fresh = () => DEMO_APPLICATIONS.map(record => ({...record}));\n'
                    "const fourth = () => ({id: 'APP-004', company: 'Example Studio', role: 'Engineering "
                    "Intern', deadline: '2027-02-10', status: 'Saved'});\n"
                    '\n'
                    "test('starter has three fictional companies, distinct IDs and complete valid records', "
                    '() => {\n'
                    '  assert.equal(DEMO_APPLICATIONS.length, 3);\n'
                    "  assert.deepEqual(DEMO_APPLICATIONS.map(record => record.company), ['Aster Labs', "
                    "'Cedar Systems', 'Orbit Works']);\n"
                    '  assert.equal(new Set(DEMO_APPLICATIONS.map(record => record.id)).size, 3);\n'
                    '  for (const record of DEMO_APPLICATIONS) assert.deepEqual(validateApplication(record), '
                    'record);\n'
                    '});\n'
                    "test('validation trims fields without changing the input', () => {\n"
                    "  const input = {...fourth(), company: '  Example Studio  ', role: ' Engineering Intern "
                    "'};\n"
                    "  assert.equal(validateApplication(input).company, 'Example Studio');\n"
                    "  assert.equal(validateApplication(input).role, 'Engineering Intern');\n"
                    "  assert.equal(input.company, '  Example Studio  ');\n"
                    '});\n'
                    "test('adding a valid application returns a new list and leaves the original unchanged', "
                    '() => {\n'
                    '  const original = fresh();\n'
                    '  const result = addApplication(original, fourth());\n'
                    '  assert.equal(result.length, 4);\n'
                    '  assert.deepEqual(result[3], fourth());\n'
                    '  assert.equal(original.length, 3);\n'
                    "  result[0].company = 'Changed copy';\n"
                    "  assert.equal(original[0].company, 'Aster Labs');\n"
                    '});\n'
                    "test('duplicate application IDs are rejected, including padded IDs', () => {\n"
                    "  assert.throws(() => addApplication(fresh(), {...fourth(), id: ' APP-001 '}), /already "
                    'exists/);\n'
                    '});\n'
                    "test('every required field must be text and company and role cannot be blank', () => {\n"
                    "  for (const field of ['id', 'company', 'role', 'deadline', 'status']) {\n"
                    '    for (const value of [undefined, null, 12]) assert.throws(() => '
                    'validateApplication({...fourth(), [field]: value}), /must be text/);\n'
                    '  }\n'
                    "  for (const field of ['company', 'role']) {\n"
                    "    for (const value of ['', '   ', 'x'.repeat(81)]) assert.throws(() => "
                    'validateApplication({...fourth(), [field]: value}), /1 to 80/);\n'
                    '  }\n'
                    '});\n'
                    "test('IDs reject spaces, lowercase, punctuation and excessive length', () => {\n"
                    "  for (const id of ['', 'app-004', 'APP 004', '<script>', 'A'.repeat(25)]) "
                    'assert.throws(() => validateApplication({...fourth(), id}), /ID must/);\n'
                    '});\n'
                    "test('deadlines require an ISO calendar date and accept a real leap day', () => {\n"
                    "  for (const deadline of ['2027-02-29', '2027-04-31', '2027-13-01', '2027-01-00', "
                    "'05/11/2027', '2027-1-05', 'tomorrow']) {\n"
                    '    assert.throws(() => validateApplication({...fourth(), deadline}), /real date/);\n'
                    '  }\n'
                    "  assert.equal(validateApplication({...fourth(), deadline: '2028-02-29'}).deadline, "
                    "'2028-02-29');\n"
                    '});\n'
                    "test('application statuses must belong to the supplied vocabulary', () => {\n"
                    "  for (const status of ['Unknown', 'applied', 'All', '']) assert.throws(() => "
                    'validateApplication({...fourth(), status}), /known status/);\n'
                    '  for (const status of STATUSES) assert.equal(validateApplication({...fourth(), '
                    'status}).status, status);\n'
                    '});\n'
                    "test('updating a known ID changes only its status and preserves the original', () => {\n"
                    '  const original = fresh();\n'
                    "  const updated = updateApplicationStatus(original, 'APP-002', 'Offer');\n"
                    "  assert.deepEqual(updated[1], {...original[1], status: 'Offer'});\n"
                    '  assert.deepEqual(updated[0], original[0]);\n'
                    "  assert.equal(original[1].status, 'Applied');\n"
                    '});\n'
                    "test('updates reject an unknown ID or an unsupported status', () => {\n"
                    "  assert.throws(() => updateApplicationStatus(fresh(), 'MISSING', 'Saved'), /Unknown "
                    'application ID/);\n'
                    "  assert.throws(() => updateApplicationStatus(fresh(), 'APP-001', 'Rejected'), /known "
                    'status/);\n'
                    '});\n'
                    "test('All and every known status filter return the expected applications', () => {\n"
                    '  assert.deepEqual(filterByStatus(fresh()), fresh());\n'
                    '  for (const status of STATUSES) assert.deepEqual(filterByStatus(fresh(), status), '
                    'fresh().filter(record => record.status === status));\n'
                    "  assert.throws(() => filterByStatus(fresh(), 'Anything'), /known status filter/);\n"
                    '});\n'
                    "test('filter results cannot mutate the source list', () => {\n"
                    '  const original = fresh();\n'
                    "  filterByStatus(original, 'Saved')[0].role = 'Changed copy';\n"
                    "  assert.equal(original[0].role, 'Frontend Intern');\n"
                    '});\n'
                    "test('summary counts all statuses and updates after add and status change', () => {\n"
                    '  assert.deepEqual(summarizeApplications(fresh()), {total: 3, counts: {Saved: 1, '
                    'Applied: 1, Interview: 1, Offer: 0, Closed: 0}});\n'
                    "  const changed = updateApplicationStatus(addApplication(fresh(), fourth()), 'APP-004', "
                    "'Offer');\n"
                    '  assert.deepEqual(summarizeApplications(changed), {total: 4, counts: {Saved: 1, '
                    'Applied: 1, Interview: 1, Offer: 1, Closed: 0}});\n'
                    '});\n'
                    "test('an empty list has empty filters and zero summary counts', () => {\n"
                    "  assert.deepEqual(filterByStatus([], 'All'), []);\n"
                    '  assert.deepEqual(summarizeApplications([]), {total: 0, counts: {Saved: 0, Applied: 0, '
                    'Interview: 0, Offer: 0, Closed: 0}});\n'
                    '});\n'
                    "test('invalid records, list containers and existing duplicate IDs are rejected', () => "
                    '{\n'
                    "  for (const value of [null, [], 'application', 1]) assert.throws(() => "
                    'validateApplication(value), /Application must/);\n'
                    "  for (const value of [null, {}, 'list']) assert.throws(() => filterByStatus(value), "
                    '/array/);\n'
                    '  assert.throws(() => summarizeApplications([DEMO_APPLICATIONS[0], '
                    'DEMO_APPLICATIONS[0]]), /unique/);\n'
                    '});\n'
                    "test('user-provided text remains literal application data', () => {\n"
                    "  const input = {...fourth(), company: '<img src=x onerror=alert(1)>', role: 'A&B "
                    "<Intern>'};\n"
                    '  const result = addApplication([], input);\n'
                    '  assert.equal(result[0].company, input.company);\n'
                    '  assert.equal(result[0].role, input.role);\n'
                    '});\n'}

# Choose an unused name for a fresh copy; never erase your edited project to restart.
PROJECT_FOLDER = 'bgs-internship-lab'
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', PROJECT_FOLDER):
    raise ValueError('Use a simple project folder name with letters, digits, hyphens or underscores.')
PROJECT = Path.cwd() / PROJECT_FOLDER
PROJECT.mkdir(exist_ok=True)
created = 0
for name, source in ASSETS.items():
    destination = PROJECT / name
    if not destination.exists():
        destination.write_text(source, encoding='utf-8')
        created += 1
REFERENCE_HASHES = {name: hashlib.sha256(ASSETS[name].encode('utf-8')).hexdigest()
                    for name in ['tracker.test.js', 'search.feature-check.mjs']}
EDITABLE_NAMES = ['tracker.js', 'app.js', 'index.html']
print(f'Node {node_version}. Project: {PROJECT}')
print(f'Created {created} files; existing edits were preserved. No packages, API keys or model calls are needed.')

def run_tests(include_feature=False):
    # Keep the baseline expectations unchanged; the new checks are a separate explicit step.
    for name, expected in REFERENCE_HASHES.items():
        actual = hashlib.sha256((PROJECT / name).read_text(encoding='utf-8').encode('utf-8')).hexdigest()
        if actual != expected:
            raise RuntimeError(f'{name} differs from the supplied reference. Preserve it and add your own checks separately.')
    filenames = ['tracker.test.js'] + (['search.feature-check.mjs'] if include_feature else [])
    result = subprocess.run([NODE, '--test-reporter=tap', '--test', *filenames], cwd=PROJECT,
                            capture_output=True, text=True, encoding='utf-8', timeout=30)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    def count(label):
        match = re.search(rf'^# {label} (\d+)', result.stdout, flags=re.MULTILINE)
        return int(match.group(1)) if match else None
    return {'returncode': result.returncode, 'tests': count('tests'), 'passed': count('pass'),
            'failed': count('fail'), 'stdout': result.stdout, 'stderr': result.stderr}

def reviewed_feature_is_current():
    # A saved review only applies to the exact source files that the student accepted.
    marker = PROJECT / 'reviewed-feature.json'
    if not marker.is_file():
        return False
    try:
        record = json.loads(marker.read_text(encoding='utf-8'))
        return all(record['sha256'].get(name) == hashlib.sha256((PROJECT / name).read_bytes()).hexdigest()
                   for name in EDITABLE_NAMES)
    except (ValueError, KeyError, TypeError):
        return False


## 2. Check the working baseline

**What:** Run the supplied functional checks before changing the app. **Why:** New work should preserve existing behavior. **Expected on a fresh project:** 16 tests, 16 pass, zero failures. The first output is saved as baseline evidence. Tests cover real add/update/filter/count behavior and input validation; they do not test browser controls.


In [ ]:
# Run the existing feature checks before proposing new work.
baseline = run_tests()
baseline_log = PROJECT / 'baseline-test-output.txt'
if not baseline_log.exists():
    baseline_log.write_text(baseline['stdout'] + baseline['stderr'], encoding='utf-8')
print('Fresh-starter expectation: 16 tests, 16 pass, 0 fail.')
print('Observed:', {key: baseline[key] for key in ['tests', 'passed', 'failed', 'returncode']})


## 3. Explore the current app

**What:** Bundle the current saved HTML, CSS and JavaScript into an inline sandboxed iframe. **Why:** This shows runtime files without starting a long-lived server or exposing a public URL. **Expected:** Three applications, working add/status/filter controls and full-list summary counts.

Try Applied, then All. Add an invented `APP-004` with a valid deadline; update a status and watch the counts. Blank fields and duplicate IDs should show a useful message. Reloading this preview restores the three examples; this teaching app keeps page data in memory. Preview interactions do not modify source files.


In [ ]:
# @title Preview the current saved app
def make_preview_html():
    # Read from disk each time so a preview reflects the student's saved changes.
    page = (PROJECT / 'index.html').read_text(encoding='utf-8')
    css = (PROJECT / 'styles.css').read_text(encoding='utf-8')
    tracker = (PROJECT / 'tracker.js').read_text(encoding='utf-8')
    app = (PROJECT / 'app.js').read_text(encoding='utf-8')
    tracker = re.sub(r'^export\s+', '', tracker, flags=re.MULTILINE)
    app = re.sub(r"^import\s+.*?from\s+['\"]\./tracker\.js['\"];?\s*$", '', app, flags=re.MULTILINE)
    if re.search(r'^\s*(?:import|export)\s', tracker + '\n' + app, flags=re.MULTILINE):
        raise RuntimeError('This preview supports the supplied two modules. Use the laptop project for additional modules.')
    script = (tracker + '\n' + app).replace('</script', '<\\/script')
    page = page.replace('<link rel="stylesheet" href="styles.css">', '<style>' + css + '</style>')
    return page.replace('<script type="module" src="app.js"></script>', '<script type="module">' + script + '</script>')

def show_preview():
    current = make_preview_html()
    display(HTML('<iframe title="Current Internship Tracker" sandbox="allow-scripts allow-forms" style="width:100%;height:1050px;border:1px solid #dce1ef;border-radius:8px" srcdoc="' + html.escape(current, quote=True) + '"></iframe>'))
    print('Preview uses current saved files. Rerun after edits; page data resets to the fictional examples.')
    return current

preview_before = show_preview()


## 4. Inspect the context before prompting

**What:** Read the actual project instructions, domain logic, UI wiring and new-feature checks. **Why:** A useful coding request names the current contract and the desired behavior. **Expected:** Explain the status vocabulary, validation, immutable list updates and separation of full-list counts from the filtered view.

`search.feature-check.mjs` describes a new function to build. It is outside the default baseline command and stays PENDING until you explicitly apply your reviewed feature. The preview also needs a labeled input and an event handler; logic tests alone cannot verify those controls.


In [ ]:
# Inspect real files instead of relying on an assistant's summary.
for filename in ['AGENTS.md', 'tracker.js', 'app.js', 'index.html', 'search.feature-check.mjs']:
    print('\n--- ' + filename + ' ---\n')
    print((PROJECT / filename).read_text(encoding='utf-8'))


## 5. Plan the search feature

**What:** Refine this short plan before editing. **Why:** Keep the request bounded and reviewable. **Expected:** Search a company **or** role substring, ignore case, trim whitespace, combine with status using both conditions, preserve full-list counts, and keep add/update/validation working.

Use the prompts in `TASK_PROMPTS.md` with an optional prepared coding client/browser assistant, or edit manually. Request a proposed diff first. Give an assistant only these fictional project files. Record the route you actually used; this notebook does not run a model client.


In [ ]:
# Edit this plan in your own words before asking for or writing a feature patch.
FEATURE_PLAN = {
    'behavior': 'Match company OR role; trim and ignore case; combine with selected status.',
    'files': ['tracker.js: add filterApplications', 'index.html: labeled search input', 'app.js: input event and combined filtering'],
    'preserve': 'Add, validation, known-status updates, status filtering, full-list counts and literal text rendering.',
    'automated_checks': 'Keep all 16 baseline checks; add the 6 supplied search checks after explicit application.',
    'browser_checks': 'Company/role match, combined status, blank/no match, add/update after search, keyboard access.',
}
print(json.dumps(FEATURE_PLAN, indent=2))
print('Next: obtain or write a focused feature proposal; review its exact diff before applying.')


## 6. Propose, review and explicitly apply

**What:** Use the source editor below to propose complete replacement text for changed files. **Why:** A proposal should be visible before any write. **Expected by default:** No files change; the new feature remains PENDING.

1. Replace the appropriate `(PROJECT / ...).read_text(...)` expression with a raw triple-quoted string containing the **complete proposed source**, for example `tracker_source = r'''...complete source...'''`. Keep unchanged sources read from disk. You can also edit files with your local editor and use this cell to record review of that current change.
2. Keep both switches False and run this cell. Read the displayed diff against the supplied starter and the additional diff against currently saved files. Check the plan, selectors, safe text rendering and preservation of baseline features.
3. Only after reviewing this exact proposal, set `I_REVIEWED_THIS_DIFF` and `APPLY_REVIEWED_FEATURE` to True and rerun this cell. A changed proposal needs another review. The cell saves only the three named source files and a fingerprint of the accepted files.

Rerunning setup preserves edits. If you change source again after recording review, the feature gate returns to PENDING until you explicitly review and apply the current source again. Reference tests must remain unchanged.


In [ ]:
# @title Source editor: review a proposal before choosing to apply it
# Replace a read_text expression with r'''complete reviewed source''' to propose an edit.
tracker_source = (PROJECT / 'tracker.js').read_text(encoding='utf-8')
app_source = (PROJECT / 'app.js').read_text(encoding='utf-8')
page_source = (PROJECT / 'index.html').read_text(encoding='utf-8')
CANDIDATE_SOURCES = {'tracker.js': tracker_source, 'app.js': app_source, 'index.html': page_source}

# First run with both False, read the exact diff, then explicitly accept the same proposal.
I_REVIEWED_THIS_DIFF = False
APPLY_REVIEWED_FEATURE = False
proposed_diff = ''
pending_write_diff = ''
for name, candidate in CANDIDATE_SOURCES.items():
    if not isinstance(candidate, str):
        raise TypeError(f'{name}: proposed source must be text.')
    proposed_diff += ''.join(difflib.unified_diff(ASSETS[name].splitlines(keepends=True), candidate.splitlines(keepends=True),
                                                fromfile='starter/' + name, tofile='proposed/' + name))
    current = (PROJECT / name).read_text(encoding='utf-8')
    pending_write_diff += ''.join(difflib.unified_diff(current.splitlines(keepends=True), candidate.splitlines(keepends=True),
                                                     fromfile='saved/' + name, tofile='proposed/' + name))
print('Proposal compared with the starter:\n' + (proposed_diff or 'No feature change proposed.'))
print('Additional writes compared with saved files:\n' + (pending_write_diff or 'No additional source writes.'))
if APPLY_REVIEWED_FEATURE and I_REVIEWED_THIS_DIFF and proposed_diff and baseline['returncode'] == 0:
    # Only these explicit project files can be written by the source editor.
    for name, candidate in CANDIDATE_SOURCES.items():
        (PROJECT / name).write_text(candidate, encoding='utf-8')
    (PROJECT / 'reviewed-feature-diff.txt').write_text(proposed_diff, encoding='utf-8')
    accepted = {name: hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() for name in EDITABLE_NAMES}
    (PROJECT / 'reviewed-feature.json').write_text(json.dumps({'feature': 'company-role-search', 'sha256': accepted}, indent=2), encoding='utf-8')
    print('Reviewed source applied. Run baseline plus new feature checks next; application alone does not prove success.')
elif reviewed_feature_is_current():
    print('Existing reviewed source preserved; verification will use its recorded review.')
else:
    print('Source files preserved. Search checks stay PENDING until a changed feature proposal is explicitly reviewed and applied.')


## 7. Verify preserved behavior and new behavior

**What:** Always rerun the 16 baseline checks. Run the 6 extra search checks only after explicit application of the currently reviewed source. **Why:** Separate working existing behavior from new work. **Expected:** Fresh Run all gives baseline PASS and search PENDING. A complete accepted feature gives 22 total passes. Inspect actual output; do not claim a pending feature passed.


In [ ]:
# Baseline preservation is checked on every run, regardless of feature progress.
final_baseline = run_tests()
(PROJECT / 'after-baseline-test-output.txt').write_text(final_baseline['stdout'] + final_baseline['stderr'], encoding='utf-8')
feature_results = None
if reviewed_feature_is_current():
    feature_results = run_tests(include_feature=True)
    (PROJECT / 'after-feature-test-output.txt').write_text(feature_results['stdout'] + feature_results['stderr'], encoding='utf-8')
    feature_status = 'PASS' if feature_results['returncode'] == 0 and feature_results['tests'] == 22 else 'NEEDS WORK'
else:
    feature_status = 'PENDING'
print('Baseline:', {key: final_baseline[key] for key in ['tests', 'passed', 'failed', 'returncode']})
print('Company/role search:', feature_status)
print('Node logic checks do not establish browser behavior; preview and check controls separately.')


## 8. Preview the current saved result

**What:** Rebuild the iframe from the files on disk. **Why:** Check the feature that was actually saved. **Expected:** Baseline preview on a default run; a search control after your accepted UI change.

After building search, try `aStEr`, `  DATA  `, `intern` plus Interview, a missing match, and blank search. Then add `APP-004`, change a status, verify full-list counts and operate controls with the keyboard. A new preview resets in-memory fictional data.


In [ ]:
# Rebuild from the accepted source on disk, not from an unaccepted proposal.
preview_after = show_preview()
print('Current feature-check status:', feature_status)


## 9. Record only browser checks you performed

**What:** Mark individual checks after carrying them out in the current preview. **Why:** A passing logic test is different evidence from operating browser controls. **Expected by default:** All False and no claimed assistant/model usage. Update the route and observations honestly; leave checks False when pending or unperformed.


In [ ]:
# Change an item only after personally observing the stated browser behavior.
UI_CHECKS = {
    'baseline_add_and_validation': False,
    'baseline_status_update_and_counts': False,
    'baseline_status_filter': False,
    'search_company_and_role_case_and_whitespace': False,
    'search_combined_with_status': False,
    'search_blank_and_no_match': False,
    'add_and_update_still_work_after_search': False,
    'keyboard_controls': False,
}
ASSISTANT_ROUTE = 'Manual editing; no model client was run by this notebook.'
UI_OBSERVATIONS = 'Pending: record what you actually saw after operating the preview.'
KNOWN_LIMITATION = 'Fictional page-memory practice; reloading resets data. No real applications or backend.'
print(json.dumps(UI_CHECKS, indent=2))
print('Actual route:', ASSISTANT_ROUTE)
print('Observations:', UI_OBSERVATIONS)


## 10. Save the observed evidence

**What:** Save actual test counts, feature status, review fingerprints and student-reported UI checks as JSON. **Why:** Another person should be able to distinguish completed work from pending checks. **Expected:** `lab-evidence.json` in the project folder. A fresh default run truthfully records baseline PASS and search PENDING.


In [ ]:
# Keep automated results separate from student-reported observations.
def result_summary(result):
    return None if result is None else {key: result[key] for key in ['tests', 'passed', 'failed', 'returncode']}
evidence = {
    'project': 'Internship Tracker / fictional workshop applications',
    'node_version': node_version,
    'baseline_before': result_summary(baseline),
    'baseline_after': result_summary(final_baseline),
    'search_feature_status': feature_status,
    'baseline_plus_search_checks': result_summary(feature_results),
    'reviewed_source_is_current': reviewed_feature_is_current(),
    'manual_ui_checks_reported_by_student': UI_CHECKS,
    'ui_observations_reported_by_student': UI_OBSERVATIONS,
    'assistant_route_reported_by_student': ASSISTANT_ROUTE,
    'known_limitation': KNOWN_LIMITATION,
}
(PROJECT / 'lab-evidence.json').write_text(json.dumps(evidence, indent=2), encoding='utf-8')
print(json.dumps(evidence, indent=2))
print('Unperformed UI checks remain False; PENDING is not a passing feature result.')


## 11. Save your project and notebook

**What:** Create a ZIP containing only the named workshop sources and evidence. **Why:** Colab runtime files are temporary. **Expected:** A clickable project archive inside the dedicated runtime folder. Save a notebook copy separately. Set `DOWNLOAD_NOW` to True only if you want Colab's browser download prompt.

On a laptop, extract the ZIP, open a terminal in its project folder and follow README.md. Explain your reviewed feature, actual checks, browser observations, assistant/manual route and one limitation in thirty seconds.


In [ ]:
# Keep the archive inside the dedicated runtime folder and include only known workshop files.
archive = PROJECT / 'BGS_Session_05_Your_Project.zip'
evidence_names = {'baseline-test-output.txt', 'after-baseline-test-output.txt', 'after-feature-test-output.txt',
                  'reviewed-feature-diff.txt', 'reviewed-feature.json', 'lab-evidence.json'}
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zipped:
    for name in sorted(set(ASSETS) | evidence_names):
        file = PROJECT / name
        if file.is_file():
            zipped.write(file, 'bgs-internship-lab/' + name)
print('Project archive:', archive.name)
display(FileLink(str(archive)))
# A Colab browser download is an explicit optional action after the ZIP exists.
DOWNLOAD_NOW = False
if DOWNLOAD_NOW:
    from google.colab import files
    files.download(str(archive))
